[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/09_teoria_decision/notebooks/01_decisiones_y_utilidad.ipynb)

# Notebook 1: Decisiones y Utilidad

En este notebook exploramos interactivamente:
1. Cómo formular un problema de decisión (matrices de pagos)
2. Funciones de utilidad y aversión al riesgo
3. Máxima utilidad esperada (MEU)
4. Valor de la información (VoI)
5. El problema del vendedor de periódicos (newsvendor)

In [ ]:
# Install dependencies (run in Colab)
# !pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue": "#2E86AB",
    "red": "#E94F37",
    "green": "#27AE60",
    "gray": "#7F8C8D",
    "orange": "#F39C12",
    "purple": "#8E44AD",
}

---
## 1. Formular una decisión

**Teoría:** [9.1 Anatomía de un problema de decisión](https://www.sonder.art/ia_p26/09_teoria_decision/01_anatomia_decision/)

Un problema de decisión tiene:
- **Estados** $S$: lo que el mundo puede ser
- **Acciones** $A$: lo que podemos hacer
- **Utilidades** $U(a, s)$: cuánto valoramos cada resultado

Lo representamos como una **matriz de pagos** (filas = acciones, columnas = estados).

### Ejercicio 1.1: Construye la matriz de pagos

**Problema del paraguas:**
- Estados: {Lluvia, Sol}
- Acciones: {Llevar paraguas, No llevar}
- Utilidades:
  - Llevar + Lluvia = 8 (seco, manos ocupadas)
  - Llevar + Sol = 5 (cargando de más)
  - No llevar + Lluvia = 1 (empapado)
  - No llevar + Sol = 10 (libre)

**Tarea:** Completa la matriz como un array de numpy.

In [ ]:
# Payoff matrix: rows = actions, columns = states
# Row 0: Llevar paraguas, Row 1: No llevar
# Col 0: Lluvia, Col 1: Sol

payoffs = np.array([
    [8, 5],   # Llevar paraguas
    [1, 10],  # No llevar paraguas
])

actions = ["Llevar paraguas", "No llevar"]
states = ["Lluvia", "Sol"]

# Verify
assert payoffs.shape == (2, 2), f"Expected (2,2), got {payoffs.shape}"
print("Matriz de pagos:")
print(f"{'':>20} {states[0]:>10} {states[1]:>10}")
for i, a in enumerate(actions):
    print(f"{a:>20} {payoffs[i, 0]:>10} {payoffs[i, 1]:>10}")

### Ejercicio 1.2: Decisión bajo certeza y maximin

In [ ]:
# Under certainty: if we KNOW it will rain, what's the best action?
best_rain = np.argmax(payoffs[:, 0])  # column 0 = rain
print(f"Si sabemos que llueve: {actions[best_rain]} (U = {payoffs[best_rain, 0]})")

# Under certainty: if we KNOW it will be sunny?
best_sun = np.argmax(payoffs[:, 1])  # column 1 = sun
print(f"Si sabemos que habrá sol: {actions[best_sun]} (U = {payoffs[best_sun, 1]})")

# Maximin: best worst case
worst_cases = payoffs.min(axis=1)  # worst outcome per action
best_maximin = np.argmax(worst_cases)
print(f"\nMaximin: {actions[best_maximin]} (peor caso = {worst_cases[best_maximin]})")
print(f"  Peor caso Llevar: {worst_cases[0]}, Peor caso No llevar: {worst_cases[1]}")

---
## 2. Utilidad y riesgo

**Teoría:** [9.2 Utilidad y preferencias racionales](https://www.sonder.art/ia_p26/09_teoria_decision/02_utilidad_preferencias/)

La relación entre dinero y utilidad determina la actitud hacia el riesgo.

### Ejercicio 2.1: Implementa tres funciones de utilidad

In [ ]:
def u_averse(x):
    """Risk-averse utility (concave)."""
    return np.sqrt(x)

def u_neutral(x):
    """Risk-neutral utility (linear)."""
    return x / 10.0

def u_seeking(x):
    """Risk-seeking utility (convex)."""
    return (x / 10.0) ** 2

# Plot all three
x = np.linspace(0, 100, 500)
fig, ax = plt.subplots()
ax.plot(x, u_averse(x), label=r"Averso: $\sqrt{x}$", color=COLORS["blue"], lw=2)
ax.plot(x, u_neutral(x), label=r"Neutral: $x/10$", color=COLORS["gray"], lw=2, ls="--")
ax.plot(x, u_seeking(x), label=r"Buscador: $(x/10)^2$", color=COLORS["red"], lw=2)
ax.set_xlabel("Riqueza")
ax.set_ylabel("Utilidad")
ax.set_title("Tres funciones de utilidad")
ax.legend()
plt.show()

### Ejercicio 2.2: Utilidad esperada de una lotería

**Lotería:** 50% de ganar \$100, 50% de ganar \$0.

**Tarea:** Calcula la EU de esta lotería bajo cada función de utilidad. Compara con U(\$50).

In [ ]:
# Lottery: 50% of $100, 50% of $0
w_high, w_low = 100.0, 0.01  # avoid sqrt(0)
p = 0.5
ew = p * w_high + (1 - p) * w_low  # expected value

for name, u_fn in [("Averso", u_averse), ("Neutral", u_neutral), ("Buscador", u_seeking)]:
    eu = p * u_fn(w_high) + (1 - p) * u_fn(w_low)  # E[U(lottery)]
    u_sure = u_fn(ew)                                # U(E[lottery])
    print(f"{name:>10}: E[U(L)] = {eu:.3f}, U(${ew:.0f}) = {u_sure:.3f}, "
          f"Jensen gap = {u_sure - eu:.3f}")

### Ejercicio 2.3: Equivalente cierto

El **equivalente cierto** (CE) es la cantidad segura que da la misma utilidad que la lotería:
$U(CE) = E[U(L)]$

**Tarea:** Encuentra el CE para cada función de utilidad usando `scipy.optimize.brentq`.

In [ ]:
for name, u_fn in [("Averso", u_averse), ("Neutral", u_neutral), ("Buscador", u_seeking)]:
    eu = p * u_fn(w_high) + (1 - p) * u_fn(w_low)
    
    # Find CE such that U(CE) = EU
    ce = brentq(lambda w: u_fn(w) - eu, 0.01, 200)
    
    risk_premium = ew - ce  # how much you'd "pay" to avoid risk
    print(f"{name:>10}: CE = ${ce:.2f}, E[L] = ${ew:.2f}, "
          f"prima de riesgo = ${risk_premium:.2f}")

### Reflexión: Utilidad y riesgo

- Para el averso al riesgo: CE < E[L]. Pagaría por eliminar incertidumbre.
- Para el neutral: CE = E[L]. Le da igual.
- Para el buscador: CE > E[L]. Pagaría por *mantener* la incertidumbre.

¿Cuál describe mejor tu comportamiento? ¿Depende de la magnitud de las apuestas?

---
## 3. Máxima Utilidad Esperada (MEU)

**Teoría:** [9.3 Decidir bajo incertidumbre](https://www.sonder.art/ia_p26/09_teoria_decision/03_decidir_bajo_incertidumbre/)

$$a^{*} = \arg\max_{a \in A} \sum_{s \in S} P(s) \cdot U(a, s)$$

### Ejercicio 3.1: MEU para el paraguas

In [ ]:
def compute_meu(payoffs, probs):
    """Compute expected utility per action and find optimal."""
    eu = payoffs @ probs  # matrix-vector product
    best = np.argmax(eu)
    return eu, best

# P(Rain) = 0.4
probs = np.array([0.4, 0.6])
eu, best = compute_meu(payoffs, probs)

for i, a in enumerate(actions):
    marker = " <-- optimal" if i == best else ""
    print(f"  EU({a}) = {eu[i]:.2f}{marker}")

### Ejercicio 3.2: Sensibilidad — varía P(Lluvia)

**Tarea:** Varía P(Lluvia) de 0 a 1 y grafica la EU de cada acción. ¿Dónde está el punto de cruce?

In [ ]:
p_rain_range = np.linspace(0, 1, 200)

eu_llevar = []
eu_no_llevar = []

for p_rain in p_rain_range:
    probs_i = np.array([p_rain, 1 - p_rain])
    eu_i, _ = compute_meu(payoffs, probs_i)
    eu_llevar.append(eu_i[0])
    eu_no_llevar.append(eu_i[1])

eu_llevar = np.array(eu_llevar)
eu_no_llevar = np.array(eu_no_llevar)

# Find crossover point
crossover_idx = np.argmin(np.abs(eu_llevar - eu_no_llevar))
p_crossover = p_rain_range[crossover_idx]

fig, ax = plt.subplots()
ax.plot(p_rain_range, eu_llevar, color=COLORS["blue"], lw=2, label="Llevar paraguas")
ax.plot(p_rain_range, eu_no_llevar, color=COLORS["red"], lw=2, label="No llevar")
ax.axvline(p_crossover, color=COLORS["gray"], ls="--", lw=1.5)
ax.scatter([p_crossover], [eu_llevar[crossover_idx]], color="black", s=80, zorder=5)
ax.text(p_crossover + 0.03, eu_llevar[crossover_idx] + 0.3,
        f"Cruce: p = {p_crossover:.3f}", fontsize=10)

# Shade regions
ax.fill_between(p_rain_range[p_rain_range <= p_crossover],
                eu_no_llevar[p_rain_range <= p_crossover],
                eu_llevar[p_rain_range <= p_crossover],
                alpha=0.1, color=COLORS["red"], label="No llevar es mejor")
ax.fill_between(p_rain_range[p_rain_range >= p_crossover],
                eu_llevar[p_rain_range >= p_crossover],
                eu_no_llevar[p_rain_range >= p_crossover],
                alpha=0.1, color=COLORS["blue"], label="Llevar es mejor")

ax.set_xlabel("P(Lluvia)")
ax.set_ylabel("Utilidad esperada")
ax.set_title("Sensibilidad: acción óptima vs P(Lluvia)")
ax.legend(fontsize=9)
plt.show()

print(f"\nPunto de cruce: P(Lluvia) ≈ {p_crossover:.3f}")
print(f"Si P(Lluvia) < {p_crossover:.3f}: no llevar")
print(f"Si P(Lluvia) > {p_crossover:.3f}: llevar")

### Reflexión: Sensibilidad

- El punto de cruce depende de los payoffs, no solo de las probabilidades.
- Si el costo de mojarse fuera mayor (U = -10 en vez de 1), ¿cómo se movería el cruce?
- ¿Qué implicación tiene esto para el valor de mejorar la predicción del clima?

---
## 4. Valor de la Información (VoI)

**Teoría:** [9.3 Valor de la Información](https://www.sonder.art/ia_p26/09_teoria_decision/03_decidir_bajo_incertidumbre/#valor-de-la-información)

$$\text{VoI}(E) = EU(\text{con info}) - EU(\text{sin info}) \geq 0$$

### Ejercicio 4.1: VPI para el test médico

**Escenario:**
- P(enfermo) = 0.1
- Tratar + Enfermo → utilidad 150
- Tratar + Sano → utilidad -50
- No tratar + Enfermo → utilidad -200
- No tratar + Sano → utilidad 0

In [ ]:
# Medical decision payoff matrix
med_payoffs = np.array([
    [150, -50],    # Treat: [sick, healthy]
    [-200, 0],     # Don't treat: [sick, healthy]
])
med_actions = ["Tratar", "No tratar"]
med_states = ["Enfermo", "Sano"]
p_sick = 0.1
med_probs = np.array([p_sick, 1 - p_sick])

# EU without info
eu_without, best_without = compute_meu(med_payoffs, med_probs)
print("Sin información:")
for i, a in enumerate(med_actions):
    marker = " <-- mejor" if i == best_without else ""
    print(f"  EU({a}) = {eu_without[i]:.1f}{marker}")

# EU with perfect info
# If sick: choose best action; if healthy: choose best action
eu_perfect = 0
for j, (state, prob) in enumerate(zip(med_states, med_probs)):
    best_for_state = np.max(med_payoffs[:, j])
    eu_perfect += prob * best_for_state
    print(f"  Si {state} (p={prob}): mejor = {best_for_state}")

vpi = eu_perfect - eu_without[best_without]
print(f"\nEU(info perfecta) = {eu_perfect:.1f}")
print(f"EU(sin info) = {eu_without[best_without]:.1f}")
print(f"VPI = {vpi:.1f}")
print(f"\n→ Un test diagnóstico perfecto vale {vpi:.1f} unidades de utilidad.")

### Ejercicio 4.2: VoI de un test imperfecto

**Tarea:** Un test tiene precisión (accuracy) del 80%. Calcula su VoI.

- P(+|enfermo) = 0.80, P(-|enfermo) = 0.20
- P(-|sano) = 0.80, P(+|sano) = 0.20

In [ ]:
def compute_voi(accuracy, p_sick, payoffs):
    """Compute VoI for a binary test with given accuracy."""
    # P(+) and P(-) via total probability
    p_pos = accuracy * p_sick + (1 - accuracy) * (1 - p_sick)
    p_neg = 1 - p_pos
    
    # Posterior: P(sick | +) and P(sick | -)
    p_sick_pos = accuracy * p_sick / p_pos if p_pos > 0 else 0
    p_sick_neg = (1 - accuracy) * p_sick / p_neg if p_neg > 0 else 0
    
    # EU given positive test
    probs_pos = np.array([p_sick_pos, 1 - p_sick_pos])
    eu_pos, _ = compute_meu(payoffs, probs_pos)
    best_pos = np.max(eu_pos)
    
    # EU given negative test
    probs_neg = np.array([p_sick_neg, 1 - p_sick_neg])
    eu_neg, _ = compute_meu(payoffs, probs_neg)
    best_neg = np.max(eu_neg)
    
    # Total EU with test
    eu_with_test = p_pos * best_pos + p_neg * best_neg
    
    # EU without test
    probs_prior = np.array([p_sick, 1 - p_sick])
    eu_no, best_no = compute_meu(payoffs, probs_prior)
    eu_without = eu_no[best_no]
    
    return eu_with_test - eu_without

# Test with 80% accuracy
acc = 0.80
voi_80 = compute_voi(acc, p_sick, med_payoffs)
print(f"VoI (accuracy = {acc*100:.0f}%) = {voi_80:.2f}")
print(f"VPI (accuracy = 100%) = {vpi:.2f}")
print(f"\nEl test imperfecto captura {voi_80/vpi*100:.1f}% del valor de info perfecta.")

In [ ]:
# Plot VoI vs accuracy
accuracies = np.linspace(0.5, 1.0, 100)
vois = [compute_voi(acc, p_sick, med_payoffs) for acc in accuracies]

fig, ax = plt.subplots()
ax.plot(accuracies * 100, vois, color=COLORS["purple"], lw=2.5)
ax.fill_between(accuracies * 100, 0, vois, alpha=0.15, color=COLORS["purple"])
ax.axhline(vpi, color=COLORS["green"], ls="--", lw=1.5, label=f"VPI = {vpi:.1f}")
ax.scatter([80], [voi_80], color=COLORS["red"], s=80, zorder=5, label=f"80% → VoI={voi_80:.1f}")

ax.set_xlabel("Precisión del test (%)")
ax.set_ylabel("Valor de la Información")
ax.set_title("VoI crece con la calidad del test")
ax.legend()
ax.set_xlim(50, 100)
ax.set_ylim(bottom=0)
plt.show()

### Reflexión: Valor de la Información

- Un test al 50% no vale nada (equivale a tirar una moneda).
- El VoI es *no lineal* — la mejora de 50→70% puede valer menos que de 90→100%.
- ¿Cuánto pagarías por un test con 80% de accuracy si VoI = lo que calculaste arriba?

---
## 5. Problema del vendedor de periódicos (Newsvendor)

**Teoría:** [9.4 Optimización estocástica](https://www.sonder.art/ia_p26/09_teoria_decision/04_optimizacion_estocastica/)

$$q^{*} = F^{-1}\left(\frac{c_u}{c_o + c_u}\right)$$

### Ejercicio 5.1: Implementa la solución óptima

In [ ]:
def newsvendor_optimal(mu, sigma, c_o, c_u):
    """Compute optimal order quantity for the newsvendor problem.
    
    Args:
        mu: mean demand
        sigma: std of demand
        c_o: overage cost per unit
        c_u: underage cost per unit
    
    Returns:
        q_star: optimal order quantity
        cr: critical ratio
    """
    cr = c_u / (c_o + c_u)
    q_star = norm.ppf(cr, mu, sigma)
    return q_star, cr

# Default parameters
mu, sigma = 50, 15
c_o, c_u = 2, 7

q_star, cr = newsvendor_optimal(mu, sigma, c_o, c_u)
print(f"Demanda ~ N({mu}, {sigma}²)")
print(f"Costo exceso c_o = ${c_o}, Costo escasez c_u = ${c_u}")
print(f"Critical ratio = {cr:.3f}")
print(f"Cantidad óptima q* = {q_star:.1f}")
print(f"E[D] = {mu}, q* - E[D] = {q_star - mu:.1f} (ordena más que la media)")

### Ejercicio 5.2: Costo esperado vs cantidad ordenada

In [ ]:
def expected_cost(q, mu, sigma, c_o, c_u, n_samples=10000):
    """Estimate expected cost via Monte Carlo."""
    np.random.seed(42)
    demands = np.random.normal(mu, sigma, n_samples)
    demands = np.maximum(demands, 0)  # demand >= 0
    
    overage = np.maximum(q - demands, 0)
    underage = np.maximum(demands - q, 0)
    costs = c_o * overage + c_u * underage
    return costs.mean()

# Sweep q values
qs = np.linspace(10, 90, 200)
costs = [expected_cost(q, mu, sigma, c_o, c_u) for q in qs]
cost_at_star = expected_cost(q_star, mu, sigma, c_o, c_u)

fig, ax = plt.subplots()
ax.plot(qs, costs, color=COLORS["blue"], lw=2.5, label="E[Costo(q)]")
ax.axvline(q_star, color=COLORS["red"], ls="--", lw=2, label=f"q* = {q_star:.1f}")
ax.scatter([q_star], [cost_at_star], color=COLORS["red"], s=100, zorder=5)
ax.axvline(mu, color=COLORS["gray"], ls=":", lw=1.5, label=f"E[D] = {mu}")

ax.set_xlabel("Cantidad ordenada (q)")
ax.set_ylabel("Costo esperado")
ax.set_title("Newsvendor: costo esperado vs cantidad")
ax.legend()
plt.show()

print(f"Costo en q* = {cost_at_star:.2f}")
print(f"Costo en E[D] = {expected_cost(mu, mu, sigma, c_o, c_u):.2f}")
print(f"Ahorro por optimizar: {expected_cost(mu, mu, sigma, c_o, c_u) - cost_at_star:.2f}")

### Ejercicio 5.3: Varía la distribución de demanda

**Tarea:** ¿Cómo cambia $q^{*}$ si la incertidumbre en la demanda aumenta?

In [ ]:
sigmas = [5, 10, 15, 20, 30]

fig, ax = plt.subplots()
for sigma_i in sigmas:
    qs_i = np.linspace(10, 90, 200)
    costs_i = [expected_cost(q, mu, sigma_i, c_o, c_u) for q in qs_i]
    q_s, _ = newsvendor_optimal(mu, sigma_i, c_o, c_u)
    ax.plot(qs_i, costs_i, lw=1.5, label=f"σ={sigma_i}, q*={q_s:.0f}")
    ax.axvline(q_s, ls=":", lw=0.8, alpha=0.5)

ax.axvline(mu, color="black", ls="--", lw=1, label=f"E[D] = {mu}")
ax.set_xlabel("Cantidad ordenada (q)")
ax.set_ylabel("Costo esperado")
ax.set_title("Efecto de la incertidumbre en la demanda")
ax.legend(fontsize=9)
plt.show()

print("\nResumen:")
for sigma_i in sigmas:
    q_s, _ = newsvendor_optimal(mu, sigma_i, c_o, c_u)
    cost_s = expected_cost(q_s, mu, sigma_i, c_o, c_u)
    print(f"  σ = {sigma_i:>2}: q* = {q_s:.1f}, costo = {cost_s:.2f}")

### Reflexión: Newsvendor

- Cuando $c_u > c_o$, ordenas **más** que la media (para protegerte contra escasez).
- Mayor incertidumbre ($\sigma$) → mayor costo esperado (aunque optimices).
- Si pudieras reducir $\sigma$ (mejor predicción), reducirías el costo. ¡Eso es el **valor de la predicción**!
- La diferencia de costo entre $\sigma_{\text{alto}}$ y $\sigma_{\text{bajo}}$ es el VoI de una mejor predicción.

---
## Resumen

En este notebook practicamos:

| Concepto | Qué hicimos |
|----------|-------------|
| Matriz de pagos | Construimos y evaluamos bajo certeza y maximin |
| Utilidad | Implementamos 3 funciones, calculamos EU y CE |
| MEU | Encontramos acción óptima y analizamos sensibilidad |
| VoI | Calculamos VPI y VoI de tests imperfectos |
| Newsvendor | Implementamos solución óptima y variamos parámetros |

**Mensaje central:** La predicción ($P(S)$) solo tiene valor si cambia la decisión ($a^{*}$).